# Deep Learning Training: Mammography BIRADS Classification

This notebook trains an EfficientNetB0-based model for classifying mammography images into BIRADS categories (1-5).

## Approach
- Transfer learning with EfficientNetB0
- Multi-class classification (5 BIRADS categories)
- Data augmentation for robustness
- Two-phase training (frozen base, then fine-tuning)


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import sys
sys.path.append('../src')

# Import Kaggle downloader
try:
    from kaggle_downloader import setup_project_data, check_kaggle_credentials
    KAGGLE_AVAILABLE = True
except ImportError:
    KAGGLE_AVAILABLE = False
    print("⚠️ Kaggle downloader not available - install kaggle package")


from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Data Preparation

In [ ]:
# Load dataset metadata
# Adjust paths based on your data structure
data_path = Path('../data')
print("Loading dataset...")

# Example structure - adjust to your actual data
# df = pd.read_csv(data_path / 'train.csv')
# Or load from directory structure
# df = create_dataframe_from_directory(data_path)

# For demonstration - you'll need to adjust this
print("Data loading structure:")
print("Expected: CSV with image paths and BIRADS labels (1-5)")
print("Or: Directory structure with folders for each BIRADS class")

# Split data
# df_train, df_temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['BIRADS'])
# df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=42, stratify=df_temp['BIRADS'])

# print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")

## Data Generators

In [ ]:
# Create data generators
IMAGE_SIZE = (256, 224)
BATCH_SIZE = 64
NUM_CLASSES = 5

# Training generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1/255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1]
)

# Validation/test generator
val_test_datagen = ImageDataGenerator(rescale=1/255.0)

# Create generators (adjust based on your data structure)
# train_gen = train_datagen.flow_from_dataframe(
#     df_train,
#     x_col='path',
#     y_col='BIRADS',
#     batch_size=BATCH_SIZE,
#     seed=42,
#     shuffle=True,
#     class_mode='sparse',
#     color_mode='rgb',
#     target_size=IMAGE_SIZE
# )

# val_gen = val_test_datagen.flow_from_dataframe(
#     df_val,
#     x_col='path',
#     y_col='BIRADS',
#     batch_size=BATCH_SIZE,
#     seed=42,
#     shuffle=False,
#     class_mode='sparse',
#     color_mode='rgb',
#     target_size=IMAGE_SIZE
# )

print("Data generators configured")
print(f"Image size: {IMAGE_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")

## Model Architecture

In [ ]:
# Import and build model
from model import build_birads_model

# Build model
model = build_birads_model(input_shape=(*IMAGE_SIZE, 3), num_classes=NUM_CLASSES)

print("Model architecture:")
print(f"Total parameters: {model.count_params():,}")
model.summary()

# Phase 1: Train with frozen base
print("\nPhase 1: Training with frozen base model...")
base_lr = 1e-4
warmup_epochs = 15

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=base_lr),
    loss='sparse_categorical_crossentropy',
    metrics=['sparse_categorical_accuracy']
)

callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint('../models/birads_phase1.h5', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-5, verbose=1)
]

# Uncomment when data is ready:
# history_phase1 = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=warmup_epochs,
#     callbacks=callbacks_phase1,
#     verbose=1
# )

## Fine-Tuning Phase

In [ ]:
# Phase 2: Fine-tuning (unfreeze some layers)
print("\nPhase 2: Fine-tuning with unfrozen layers...")

# Unfreeze top 20% of layers
num_layers = len(model.layers)
trainable_from = int(num_layers * 0.8)

for i, layer in enumerate(model.layers):
    if i >= trainable_from:
        layer.trainable = True
    else:
        layer.trainable = False

# Recompile with lower learning rate
fine_lr = 1e-4
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=fine_lr),
    loss='sparse_categorical_crossentropy',
    metrics=['sparse_categorical_accuracy']
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('../models/birads_final.h5', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    TensorBoard(log_dir='../logs', histogram_freq=1)
]

# Uncomment when data is ready:
# history_phase2 = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=100,
#     initial_epoch=warmup_epochs,
#     callbacks=callbacks_phase2,
#     verbose=1
# )

print("Fine-tuning configuration complete")

In [ ]:
# Evaluate on test set
# test_gen = val_test_datagen.flow_from_dataframe(
#     df_test,
#     x_col='path',
#     y_col='BIRADS',
#     batch_size=BATCH_SIZE,
#     seed=42,
#     shuffle=False,
#     class_mode='sparse',
#     color_mode='rgb',
#     target_size=IMAGE_SIZE
# )

# test_results = model.evaluate(test_gen, verbose=1)
# print(f"\nTest Loss: {test_results[0]:.4f}")
# print(f"Test Accuracy: {test_results[1]:.4f}")

# Make predictions
# test_gen.reset()
# predictions = model.predict(test_gen, verbose=1)
# y_pred = np.argmax(predictions, axis=1)
# y_true = df_test['BIRADS'].values[:len(y_pred)]

# Classification report
# print("\nClassification Report:")
# print(classification_report(y_true, y_pred, target_names=[f'BIRADS {i+1}' for i in range(5)]))

# Confusion matrix
# cm = confusion_matrix(y_true, y_pred)
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
#             xticklabels=[f'BIRADS {i+1}' for i in range(5)],
#             yticklabels=[f'BIRADS {i+1}' for i in range(5)])
# plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
# plt.ylabel('True Label', fontsize=12)
# plt.xlabel('Predicted Label', fontsize=12)
# plt.tight_layout()
# plt.show()

# Cohen's Kappa
# kappa_linear = cohen_kappa_score(y_true, y_pred)
# kappa_quadratic = cohen_kappa_score(y_true, y_pred, weights='quadratic')
# print(f"\nCohen's Kappa (Linear): {kappa_linear:.4f}")
# print(f"Cohen's Kappa (Quadratic): {kappa_quadratic:.4f}")

print("Evaluation code ready - uncomment when data is loaded")


## Training History Visualization


In [ ]:
# Plot training history (uncomment when training is complete)
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# 
# # Combine histories
# if 'history_phase1' in locals() and 'history_phase2' in locals():
#     history = {
#         'loss': history_phase1.history['loss'] + history_phase2.history['loss'],
#         'val_loss': history_phase1.history['val_loss'] + history_phase2.history['val_loss'],
#         'sparse_categorical_accuracy': history_phase1.history['sparse_categorical_accuracy'] + history_phase2.history['sparse_categorical_accuracy'],
#         'val_sparse_categorical_accuracy': history_phase1.history['val_sparse_categorical_accuracy'] + history_phase2.history['val_sparse_categorical_accuracy']
# }
# 
# # Loss
# axes[0].plot(history['loss'], label='Training Loss', linewidth=2)
# axes[0].plot(history['val_loss'], label='Validation Loss', linewidth=2)
# axes[0].set_title('Model Loss', fontsize=14, fontweight='bold')
# axes[0].set_xlabel('Epoch', fontsize=12)
# axes[0].set_ylabel('Loss', fontsize=12)
# axes[0].legend()
# axes[0].grid(True, alpha=0.3)
# 
# # Accuracy
# axes[1].plot(history['sparse_categorical_accuracy'], label='Training Accuracy', linewidth=2)
# axes[1].plot(history['val_sparse_categorical_accuracy'], label='Validation Accuracy', linewidth=2)
# axes[1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
# axes[1].set_xlabel('Epoch', fontsize=12)
# axes[1].set_ylabel('Accuracy', fontsize=12)
# axes[1].legend()
# axes[1].grid(True, alpha=0.3)
# 
# plt.tight_layout()
# plt.show()

print("Visualization code ready")
